## Imports

In [1]:
import numpy as np
import cv2
import plotly.express as px
from scipy.linalg import circulant
import plotly.graph_objects as go

from utils import *

In [ ]:
def createCircle(r, c, K):
    """create a circle of radius r centered at c with K points"""
    alphas = np.linspace(0, 2 * np.pi, K, endpoint=False)
    return [(int(r*np.cos(alpha)) + c[0], int(r*np.sin(alpha)) + c[1])  for alpha in alphas]

def createDMatrix(K, alpha, beta):
    """"create the D matrix for the snake algorithm"""
    c2 = np.zeros(K)
    c2[0] = -2; c2[1] = 1; c2[-1] = 1
    D2 = circulant(c2)


    c4 = np.zeros(K)
    c4[0] = 6; c4[1] = -4; c4[2] = 1;c4[-1] = -4; c4[-2] = 1
    D4 = circulant(c4)

    return alpha * D2 - beta * D4


def processSnakeBaloon(img, alpha, beta, gamma, k, dt, initialSnake, Niter=0, snapShots=0):
    """Execute the snake algorithm with balloon forces on the given image and initial snake"""
    img = img.copy()
    snake = np.asarray(initialSnake, dtype=float).copy()
    K = snake.shape[0]

    gradImgy, gradImgx = np.gradient(img)

    grad = gradImgx**2 + gradImgy**2
    gradNormy, gradNormx = np.gradient(grad)

    D = createDMatrix(K, alpha, beta)
    A = np.linalg.inv(np.eye(K) - dt * D)

    Niter = int(Niter)
    E = np.zeros((Niter, K, 2))

    snapshot_step = max(1, int(round(Niter * snapShots))) if snapShots else Niter + 1
    x_snap_shots = [snake[:, 0].copy()]
    y_snap_shots = [snake[:, 1].copy()]

    x = snake[:, 0].copy()
    y = snake[:, 1].copy()

    for i in range(Niter):
        xi = np.clip(x.astype(int), 0, gradNormx.shape[1] - 1)
        yi = np.clip(y.astype(int), 0, gradNormx.shape[0] - 1)

        force_x = gradNormx[yi, xi]
        force_y = gradNormy[yi, xi]

        tx = -np.roll(x, -1) + np.roll(x, 1)
        ty = -np.roll(y, -1) + np.roll(y, 1)
        norm_t = np.sqrt(tx**2 + ty**2) + 1e-8
        nx = -ty / norm_t
        ny =  tx / norm_t

        E[i, :, 0] = D @ x - gamma * force_x - k * nx
        E[i, :, 1] = D @ y - gamma * force_y - k * ny

        x = A @ (x + dt * (gamma * force_x + k * nx))
        y = A @ (y + dt * (gamma * force_y + k * ny))

        if (i + 1) % snapshot_step == 0 or i == Niter - 1:
            x_snap_shots.append(x.copy())
            y_snap_shots.append(y.copy())

    return np.array(x_snap_shots), np.array(y_snap_shots), E

In [31]:
img = cv2.imread('imagesTP/im1.png', 0)
img = cv2.normalize(img, None, 0, 1.0, cv2.NORM_MINMAX, dtype=cv2.CV_32F)
H, W = img.shape
K = 300 # Nb points for the snake
circle = np.array(createCircle(10, (130, 130), K)) # Snake initialisation

alpha = 10
beta = .02
gamma = 30
k = 2
dt = .1
Niter = 20_000

x_snapshots, y_snapshots, E = processSnakeBaloon(img, alpha, beta, gamma,k, dt, initialSnake=circle, Niter=Niter, snapShots=1/100)
showAnim(img, x_snapshots, y_snapshots, 100)
displayEnergy(E)

array([4.98162489, 2.99848449, 2.42760119, ..., 4.31656235, 4.18151006,
       4.38136627], shape=(20000,))